# Smoke-тест файнтюна: Qwen3-14B + LoRA (SEQ_CLS)

Цель — за ~3-5 минут на GPU проверить, что пайплайн файнтюна поднимается:
1. Данные грузятся, `label2id` детерминированный, 36 классов.
2. База модели + LoRA-адаптер корректно оборачиваются, `print_trainable_parameters()` показывает `score.*` в trainable.
3. Trainer делает 100 шагов без OOM.
4. `evaluate()` пишет `results/finetune_results.csv` и `preds_lora_qwen3_14b.csv`.

**Требования к Runtime:** L4 (24GB) или A100 (40/80GB). На T4 14B в bf16 не влезет.

**Метод патчится "на лету":** `max_steps=100`, `num_train_epochs=1`, `save_strategy="no"` — чтобы не ждать полную эпоху и не писать чекпоинты.

## 1. Клонирование репозитория и установка зависимостей

Если репозиторий приватный — замени URL на `https://<PAT>@github.com/KVTur23/Mifi_VKR.git`.

In [ ]:
!git clone --branch fine-tune https://github.com/KVTur23/Mifi_VKR.git /content/vkr 2>&1 | tail -3
%cd /content/vkr/code
!pip install -q -r requirements.txt 2>&1 | tail -5

In [ ]:
!pip install -q -U 'peft>=0.19.0'

In [ ]:
import peft; print(peft.__version__)

In [ ]:
import torch, transformers, peft, datasets
print("torch:       ", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:        ", peft.__version__)
print("datasets:    ", datasets.__version__)


import json, sys, os

os.chdir("/content/vkr/code")
sys.path.insert(0, "/content/vkr/code")

pc_raw = json.load(open("/content/vkr/code/config_models/pipeline_config.json"))
pipeline_cfg = {"gpu_name": GPU, "finetune": pc_raw["finetune"]}

print("GPU profile:", GPU, "→", pipeline_cfg["finetune"][GPU])
print("Common:      ", pipeline_cfg["finetune"]["common"])
print("cwd:         ", os.getcwd())

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({vram:.1f} GB)")
else:
    raise RuntimeError("CUDA недоступна. Смените Runtime на GPU (L4/A100).")

# Выбери профиль под GPU — эта строка единственная, что ты правишь
GPU = "A100_40"  # варианты: "T4", "L4", "A100_40", "A100_80", "H100"

## 3. Сборка `pipeline_cfg`

Минимальный контракт для `SeqClsRunner`:
- `gpu_name` — имя профиля (строка).
- `finetune` — вся секция из `pipeline_config.json` (все профили + `common`).

Когда будет оркестратор из Фазы 4, эту сборку сделает `orchestrator.run_finetune`.

In [ ]:
import json, sys
sys.path.insert(0, "/content/vkr/code")

pc_raw = json.load(open("config_models/pipeline_config.json"))
pipeline_cfg = {"gpu_name": GPU, "finetune": pc_raw["finetune"]}

print("GPU profile:", GPU, "→", pipeline_cfg["finetune"][GPU])
print("Common:      ", pipeline_cfg["finetune"]["common"])

## 4. Prepare: загрузка данных, модели, PEFT-адаптера

`runner.prepare()` выполняет:
1. Грузит `data_after_stage3.csv` (train), `data_test.csv` (test), `train_after_eda.csv` (для подсчёта оригиналов на класс → группы A/B/C).
2. Строит детерминированный `label2id` (через `sorted()`), проверяет что число классов = `num_labels=36`.
3. Загружает Qwen3-14B + токенизатор (`padding_side="left"`, `pad_token=eos_token`).
4. Токенизирует train/test (truncation до `max_seq` из GPU-профиля, динамический паддинг в коллаторе).
5. Оборачивает модель LoRA (`get_peft_model`), вызывает `print_trainable_parameters()` и проверяет что `score.*` есть в trainable.

**Что смотреть в выводе:**
- Строку `trainable params: X || all params: Y || trainable%: Z` — должна быть доля порядка ~0.1-1%.
- Отсутствие ошибки `Classification head 'score' не попала в trainable params` (если упало — у Qwen3 голова названа иначе и надо поправить `modules_to_save` в конфиге).

In [ ]:
from src.finetune.trainer_base import SeqClsRunner
from src.finetune.run_lora import CONFIG_PATH

runner = SeqClsRunner(CONFIG_PATH, pipeline_cfg)
runner.prepare()

print()
print("=" * 60)
print("PREPARE OK")
print("=" * 60)
print("run_key:        ", runner.run_key)
print("output_dir:     ", runner.output_dir)
print("num classes:    ", len(runner.label2id))
print("trainable params:", f"{runner.trainable_params:,}")
print("train/test size:", len(runner.train_ds), "/", len(runner.eval_ds))
print("max_seq_length: ", runner.cfg["max_seq_length"])

## 5. Smoke-патч training_params

Применяется **до** `runner.train()` — `TrainingArguments` внутри `train()` читает `self.cfg["training_params"]`, так что патч подхватится.

- `max_steps=100` — жёстко обрезает число шагов оптимизатора, даже внутри первой эпохи.
- `save_strategy="no"` — чекпоинты не пишем, не засоряем диск.
- `load_best_model_at_end=False` — совместимо с `save_strategy="no"`.
- `eval_steps=50` — две оценки за 100 шагов, видно динамику macro_f1.

In [ ]:
tp = runner.cfg["training_params"]
tp["num_train_epochs"] = 1
tp["max_steps"] = 100
tp["eval_strategy"] = "steps"
tp["eval_steps"] = 50
tp["save_strategy"] = "no"
tp["load_best_model_at_end"] = False
tp["logging_steps"] = 10

print("Патч применён:")
for k in ("num_train_epochs", "max_steps", "eval_strategy", "eval_steps",
         "save_strategy", "load_best_model_at_end", "logging_steps",
         "per_device_train_batch_size", "gradient_accumulation_steps",
         "learning_rate", "bf16", "fp16"):
    print(f"  {k}: {tp.get(k)}")

## 6. Train: 100 шагов

Trainer сделает 100 шагов с двумя eval-чек-поинтами на шагах 50 и 100.
По итогам `runner.train()` сохранит адаптер в `Data/finetune_checkpoints/lora_qwen3_14b/`
(даже при `save_strategy="no"` — финальное сохранение делаем мы сами через `save_adapter`).

In [ ]:
runner.train()

print()
print(f"train_time_sec: {runner.train_time_sec:.1f}")
print(f"artefacts:       {runner.output_dir}")

## 7. Evaluate на test

Прогоняет весь test через адаптер, считает `balanced_accuracy`, `macro_f1`, `f1_group_{A,B,C}`,
пишет `results/preds_lora_qwen3_14b.csv` и строку в `results/finetune_results.csv`.

**Sanity-check:** `macro_f1` должен быть заметно выше случайного 1/36 ≈ 0.028.
На 100 шагах реалистично ожидать 0.05-0.20 — это нормально для smoke, полноценные 5 эпох потом.

In [ ]:
from src.finetune.evaluate_finetuned import evaluate

metrics = evaluate(
    adapter_dir=str(runner.output_dir),
    config_path=runner.config_path,
    pipeline_cfg=pipeline_cfg,
    run_key=runner.run_key,
)

## 8. Артефакты

Проверка того, что CSV появились в правильной схеме проекта.

In [ ]:
import pandas as pd

print("=== results/finetune_results.csv ===")
print(pd.read_csv("results/finetune_results.csv").to_string(index=False))

print()
print(f"=== results/preds_{runner.run_key}.csv (первые 5) ===")
print(pd.read_csv(f"results/preds_{runner.run_key}.csv").head().to_string(index=False))

## Что прислать обратно при дебаге

Если что-то упало — сохрани и отдай:
1. Полный вывод ячейки §4 (`runner.prepare()`) — там `print_trainable_parameters()` и сам трейс.
2. Вывод ячейки §6 (`runner.train()`) — первые логи trainer'а и OOM/любую ошибку.
3. Содержимое `results/finetune_results.csv`, если ячейка §7 прошла.

**Типичные грабли** (§8 плана):
- `Classification head 'score' не попала в trainable` — значит у Qwen3 голова называется иначе. Правим `modules_to_save` во всех 4 `finetune_configs/*.json`.
- `Trainer.__init__() got unexpected keyword argument 'tokenizer'` — в transformers 5.x переименовано в `processing_class`. Правим `trainer_base.py`.
- OOM — уменьшаем `per_device_batch` / `max_seq` в pipeline_config или переключаемся на QLoRA.